In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def plot_cylindrical_shearing_box():
    fig = plt.figure(figsize=(14, 10), dpi=100)
    ax = fig.add_subplot(111, projection='3d')

    # --- 1. Define Cylindrical Geometry ---
    r_outer = 10.0  # Outer cylinder radius
    r_center = 7.0  # Reference radius for the shearing box (midpoint)
    theta_center = np.pi / 4  # 45 degrees in polar coordinates
    
    # --- 2. Draw Global Cylindrical Context (OUTER CYLINDER ONLY) ---
    theta_full = np.linspace(0, 2*np.pi, 100)
    z_cyl = np.linspace(-2.5, 2.5, 6)  # Horizontal circles at different z
    
    # Outer cylinder horizontal circles - more visible
    for z in z_cyl:
        ax.plot(r_outer * np.cos(theta_full), r_outer * np.sin(theta_full), 
               z * np.ones_like(theta_full), 
               color='gray', linestyle='-', alpha=0.5, linewidth=2)
    
    # Radial lines on outer cylinder - showing meridians
    n_meridians = 12
    for theta in np.linspace(0, 2*np.pi, n_meridians, endpoint=False):
        ax.plot([r_outer*np.cos(theta)]*2, 
               [r_outer*np.sin(theta)]*2, 
               [-2.5, 2.5], color='gray', linestyle='-', alpha=0.5, linewidth=2)
    
    # Central axis with rotation arrow
    ax.plot([0, 0], [0, 0], [-3.5, 3.5], color='black', linewidth=3.0, linestyle='--')
    ax.text(0, 0, 4.3, r'$\Omega_0$', fontsize=20, ha='center', weight='bold')
    
    # Rotation indicator (curved arrow) - more prominent
    t_rot = np.linspace(0, 1.5*np.pi, 30)
    r_rot = 2.0
    ax.plot(r_rot * np.cos(t_rot), r_rot * np.sin(t_rot), 
           3.5*np.ones_like(t_rot), color='black', lw=3)
    ax.quiver(r_rot*np.cos(t_rot[-1]), r_rot*np.sin(t_rot[-1]), 3.5, 
              -r_rot*np.sin(t_rot[-1])*0.4, r_rot*np.cos(t_rot[-1])*0.4, 0, 
              length=1.0, color='black', arrow_length_ratio=0.35, linewidth=2.5)

    # --- 3. Define Local Cartesian Shearing Box ---
    # Box dimensions in LOCAL coordinates
    L_x = 4.0  # Azimuthal extent (local x)
    L_y = 4.0  # Radial extent (local y)
    L_z = 4.0  # Vertical extent (local z)
    
    # Position of box center in cylindrical coords (at 45 degrees)
    box_center_cyl = np.array([r_center * np.cos(theta_center), 
                               r_center * np.sin(theta_center), 0])
    
    # Local coordinate system (tangent to cylinder at center)
    x_hat = np.array([-np.sin(theta_center), np.cos(theta_center), 0])  # Azimuthal
    y_hat = np.array([np.cos(theta_center), np.sin(theta_center), 0])   # Radial (points toward viewer at 45°)
    z_hat = np.array([0, 0, 1])  # Vertical
    
    # Box vertices in local coordinates
    box_corners_local = np.array([
        [-L_x/2, -L_y/2, -L_z/2], [L_x/2, -L_y/2, -L_z/2],
        [L_x/2, L_y/2, -L_z/2], [-L_x/2, L_y/2, -L_z/2],
        [-L_x/2, -L_y/2, L_z/2], [L_x/2, -L_y/2, L_z/2],
        [L_x/2, L_y/2, L_z/2], [-L_x/2, L_y/2, L_z/2]
    ])
    
    # Transform to global coordinates
    box_corners_global = np.zeros_like(box_corners_local)
    for i, corner in enumerate(box_corners_local):
        box_corners_global[i] = (box_center_cyl + 
                                corner[0] * x_hat + 
                                corner[1] * y_hat + 
                                corner[2] * z_hat)
    
    # Z-level of top surface
    z_top = L_z/2
    
    # Draw box edges with thicker lines
    edges = [
        [0, 1], [1, 2], [2, 3], [3, 0],  # Bottom face
        [4, 5], [5, 6], [6, 7], [7, 4],  # Top face
        [0, 4], [1, 5], [2, 6], [3, 7]   # Vertical edges
    ]
    
    for edge in edges:
        points = box_corners_global[edge]
        ax.plot3D(*points.T, color='black',linestyle='--', linewidth=2.0)
    
    # Highlight front face with transparency
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    front_face = [box_corners_global[i] for i in [2, 3, 7, 6]]  # Face toward viewer
    ax.add_collection3d(Poly3DCollection([front_face], alpha=0.2, facecolor='lightblue', edgecolor='none'))
    
    # Highlight top face with slight transparency
    top_face = [box_corners_global[i] for i in [4, 5, 6, 7]]
    ax.add_collection3d(Poly3DCollection([top_face], alpha=0.15, facecolor='lightyellow', edgecolor='none'))

    # --- 4. Draw Radial Line (45-degree meridian) AT TOP SURFACE LEVEL ---
    ax.plot([0, r_outer*np.cos(theta_center)], 
           [0, r_outer*np.sin(theta_center)], 
           [z_top, z_top], color='orange', linestyle='-', alpha=0.8, linewidth=3.5, 
           label=r'$\theta = 45°$ meridian')

    # --- 5. Draw Shear Profile (AT TOP SURFACE LEVEL) ---
    # Show velocity arrows at different radial positions
    n_arrows = 7
    y_positions = np.linspace(-L_y/2, L_y/2, n_arrows)
    
    for y_local in y_positions:
        # Position in global coordinates at top surface
        pos_global = box_center_cyl + y_local * y_hat + z_top * z_hat
        
        # Shear velocity: u = S * y (differential rotation)
        # where S is the shear rate
        shear_rate = 0.65
        u_local = shear_rate * y_local
        u_global = u_local * x_hat
        
        # Draw velocity arrow - thicker and more visible
        ax.quiver(pos_global[0], pos_global[1], pos_global[2],
                 u_global[0], u_global[1], 0,
                 length=1.0, normalize=False, color='saddlebrown', 
                 arrow_length_ratio=0.25, linewidth=2.0)
    
    # Draw line connecting arrow tips to show linear shear AT TOP SURFACE
    tip_positions = []
    for y_local in y_positions:
        pos_global = box_center_cyl + y_local * y_hat + z_top * z_hat
        u_local = shear_rate * y_local
        u_global = u_local * x_hat
        tip = pos_global + u_global
        tip_positions.append(tip)
    tip_positions = np.array(tip_positions)
    ax.plot(tip_positions[:, 0], tip_positions[:, 1], 
           tip_positions[:, 2], color='saddlebrown', linewidth=2.0, linestyle='-', alpha=1.0)

    # --- 6. Draw Vortex Structure ON TOP SURFACE ---
    # Elliptical vortex in the shearing box
    vortex_center_local = np.array([0.3, 0.0, z_top+2.0])  # Now at top surface
    vortex_center_global = (box_center_cyl + 
                           vortex_center_local[0] * x_hat + 
                           vortex_center_local[1] * y_hat + 
                           vortex_center_local[2] * z_hat)
    
    # Create ellipse in local coordinates
    t_ellipse = np.linspace(0, 2*np.pi, 100)
    a_vortex = 0.8  # Semi-major axis (in x direction)
    b_vortex = 0.6  # Semi-minor axis (in y direction)
    
    ellipse_local = np.column_stack([
        a_vortex * np.cos(t_ellipse),
        b_vortex * np.sin(t_ellipse),
        np.zeros_like(t_ellipse)  # Stays on the z_top plane
    ])
    
    # Transform to global coordinates
    ellipse_global = np.zeros_like(ellipse_local)
    for i, point in enumerate(ellipse_local):
        ellipse_global[i] = (vortex_center_global + 
                            point[0] * x_hat + 
                            point[1] * y_hat)  # No z component in the perturbation
    
    ax.plot(ellipse_global[:, 0], ellipse_global[:, 1], ellipse_global[:, 2],
           color='darkblue', linewidth=1.5)
    
    # Add circulation arrows on vortex
    arrow_indices = [20, 70]
    for idx in arrow_indices:
        direction = ellipse_global[idx+1] - ellipse_global[idx]
        ax.quiver(ellipse_global[idx, 0], ellipse_global[idx, 1], ellipse_global[idx, 2],
                 direction[0], direction[1], direction[2],
                 length=0.5, normalize=True, color='darkblue', arrow_length_ratio=0.35, linewidth=3)

    # --- 7. Draw Local Coordinate System ON TOP SURFACE ---
    arrow_length = 2.5
    # Place arrows at the center of the top surface
    arrow_origin = box_center_cyl + np.array([0, 0, z_top])
    
    # Local x (azimuthal) - RED
    ax.quiver(arrow_origin[0], arrow_origin[1], arrow_origin[2],
             x_hat[0], x_hat[1], x_hat[2],
             length=arrow_length, color='red', arrow_length_ratio=0.15, linewidth=3)
    ax.text(arrow_origin[0] + arrow_length*x_hat[0]*1.4, 
           arrow_origin[1] + arrow_length*x_hat[1]*1.2,
           arrow_origin[2] - arrow_length*x_hat[2]*1.2, r'$x$ (local azimuthal $\hat{\theta}$)', 
           color='red', fontsize=16)
    
    # Local y (radial) - GREEN
    ax.quiver(arrow_origin[0], arrow_origin[1], arrow_origin[2],
             y_hat[0], y_hat[1], y_hat[2],
             length=arrow_length, color='green', arrow_length_ratio=0.15, linewidth=3)
    ax.text(arrow_origin[0] + arrow_length*y_hat[0]*1.2,
           arrow_origin[1] + arrow_length*y_hat[1]*1.2,
           arrow_origin[2], r'$y$ (local radial $\hat{r}$)', 
           color='green', fontsize=16)
    
    # Local z (vertical) - PURPLE
    ax.quiver(arrow_origin[0], arrow_origin[1], arrow_origin[2],
             0, 0, 1,
             length=arrow_length, color='purple', arrow_length_ratio=0.15, linewidth=3)
    ax.text(arrow_origin[0], arrow_origin[1], 
           arrow_origin[2] + arrow_length*1.2,
           r'$z$ (vertical)', color='purple', fontsize=16)

    # --- 8. Formatting ---
    ax.set_xlim(-13, 13)
    ax.set_ylim(-13, 13)
    ax.set_zlim(-4, 6)
    ax.set_box_aspect([1, 1, 0.65])
    
    # Clean background
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('w')
    ax.yaxis.pane.set_edgecolor('w')
    ax.zaxis.pane.set_edgecolor('w')
    ax.grid(False)
    ax.set_axis_off()
    
    # Set view angle - adjusted for clearer view
    ax.view_init(elev=30, azim=-50)
    
    plt.tight_layout()
    return fig

# Generate the plot
fig = plot_cylindrical_shearing_box()
plt.show()